In [1]:
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit import transpile
import re

In [2]:
def parse_quipper_ascii(filename):
    ops = []
    with open(filename) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("Comment") or line.startswith("Inputs") or line.startswith("Outputs"):
                continue

            # Hadamard
            if line.startswith('QGate["H"]'):
                idx = int(line.split("(")[-1].split(")")[0])
                ops.append(("H", [idx], []))

            # NOT / Controlled-NOT / Toffoli
            elif line.startswith('QGate["not"]'):
                target = int(line.split("(")[-1].split(")")[0])
                if "controls" in line:
                    ctrl_str = line.split("controls=[+")[1].split("]")[0]
                    ctrls = [int(c) for c in ctrl_str.split(",+")]
                    if len(ctrls) == 1:
                        ops.append(("CNOT", [ctrls[0], target], []))
                    elif len(ctrls) == 2:
                        ops.append(("Toffoli", ctrls + [target], []))
                else:
                    ops.append(("X", [target], []))

            # Controlled-Z
            # elif line.startswith('QGate["Z"]'):
            #     target = int(line.split("(")[-1].split(")")[0])
            #     ctrl_str = line.split("controls=[+")[1].split("]")[0]
            #     ctrls = [int(c) for c in ctrl_str.split(",+")]
            #     ops.append(("CZ", ctrls + [target], []))
            elif line.startswith('QGate["Z"]'):
                target = int(line.split("(")[-1].split(")")[0])
                if "controls" in line:
                    ctrl_str = line.split("controls=[+")[1].split("]")[0]
                    ctrls = [int(c) for c in ctrl_str.split(",+")]
                    if len(ctrls) == 1:
                        ops.append(("CZ", ctrls + [target], []))
                    elif len(ctrls) == 2:
                        ops.append(("CCZ", ctrls + [target], []))
                else:
                    # plain Z gate with no controls
                    ops.append(("Z", [target], []))

            # RZ rotation
            elif line.startswith('QRot["exp(-i%Z)"'):
                angle_match = re.search(r"[-+]?\d*\.\d+|\d+", line)
                angle = float(angle_match.group()) if angle_match else 0.0
                idx = int(line.split("(")[-1].split(")")[0])
                ops.append(("RZ", [idx], [angle]))

            # SWAP
            elif line.startswith('QGate["swap"]'):
                parts = line.split("(")[-1].split(")")[0].split(",")
                ws = [int(p) for p in parts]
                ops.append(("SWAP", ws, []))

    return ops


def quipper_to_qnode(filename, wires=None):
    ops = parse_quipper_ascii(filename)
    if wires is None:
        max_wire = max(max(ws) for _, ws, _ in ops)
        wires = max_wire + 1

    dev = qml.device("default.qubit", wires=wires)

    @qml.qnode(dev)
    def circuit():
        for name, ws, params in ops:
            if name == "H":
                qml.Hadamard(wires=ws[0])
            elif name == "X":
                qml.PauliX(wires=ws[0])
            elif name == "CNOT":
                qml.CNOT(wires=ws)
            elif name == "Toffoli":
                qml.Toffoli(wires=ws)
            elif name == "CZ":
                # general controlled-Z
                ctrls, target = ws[:-1], ws[-1]
                qml.ctrl(qml.PauliZ, control=ctrls)(wires=target)
            elif name == "RZ":
                qml.RZ(params[0], wires=ws[0])
            elif name == "SWAP":
                qml.SWAP(wires=ws)
        return qml.state()

    return circuit

In [3]:
# with the decomposition of multi-controlled gates
def parse_quipper_ascii_extended(filename):
    ops = []
    with open(filename) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("Comment") or line.startswith("Inputs") or line.startswith("Outputs"):
                continue

            # Hadamard
            if line.startswith('QGate["H"]'):
                idx = int(line.split("(")[-1].split(")")[0])
                ops.append(("H", [idx], []))

            # NOT / Controlled-NOT / Toffoli
            elif line.startswith('QGate["not"]'):
                target = int(line.split("(")[-1].split(")")[0])
                if "controls" in line:
                    ctrl_str = line.split("controls=[+")[1].split("]")[0]
                    ctrls = [int(c) for c in ctrl_str.split(",+")]
                    if len(ctrls) == 1:
                        ops.append(("CNOT", [ctrls[0], target], []))
                    elif len(ctrls) == 2:
                        ops.append(("Toffoli", ctrls + [target], []))
                else:
                    ops.append(("X", [target], []))

            # Controlled-Z / CCZ
            # elif line.startswith('QGate["Z"]'):
            #     target = int(line.split("(")[-1].split(")")[0])
            #     ctrl_str = line.split("controls=[+")[1].split("]")[0]
            #     ctrls = [int(c) for c in ctrl_str.split(",+")]
            #     if len(ctrls) == 1:
            #         ops.append(("CZ", ctrls + [target], []))
            #     elif len(ctrls) == 2:
            #         ops.append(("CCZ", ctrls + [target], []))
            elif line.startswith('QGate["Z"]'):
                target = int(line.split("(")[-1].split(")")[0])
                if "controls" in line:
                    ctrl_str = line.split("controls=[+")[1].split("]")[0]
                    ctrls = [int(c) for c in ctrl_str.split(",+")]
                    if len(ctrls) == 1:
                        ops.append(("CZ", ctrls + [target], []))
                    elif len(ctrls) == 2:
                        ops.append(("CCZ", ctrls + [target], []))
                else:
                    # plain Z gate with no controls
                    ops.append(("Z", [target], []))
            # RZ rotation
            elif line.startswith('QRot["exp(-i%Z)"'):
                angle_match = re.search(r"[-+]?\d*\.\d+|\d+", line)
                angle = float(angle_match.group()) if angle_match else 0.0
                idx = int(line.split("(")[-1].split(")")[0])
                ops.append(("RZ", [idx], [angle]))

            # SWAP
            elif line.startswith('QGate["swap"]'):
                parts = line.split("(")[-1].split(")")[0].split(",")
                ws = [int(p) for p in parts]
                ops.append(("SWAP", ws, []))

    return ops


def quipper_to_qnode_extended(filename, wires=None):
    ops = parse_quipper_ascii_extended(filename)
    if wires is None:
        max_wire = max(max(ws) for _, ws, _ in ops)
        wires = max_wire + 1

    dev = qml.device("default.qubit", wires=wires)

    @qml.qnode(dev)
    def circuit():
        for name, ws, params in ops:
            if name == "H":
                qml.Hadamard(wires=ws[0])
            elif name == "X":
                qml.PauliX(wires=ws[0])
            elif name == "CNOT":
                qml.CNOT(wires=ws)
            elif name == "Toffoli":
                # Equivalence formula for Toffoli
                target = ws[-1]
                qml.Hadamard(wires=target)
                qml.RZ(np.pi/4, wires=target)
                qml.CNOT(wires=[ws[1], target])
                qml.RZ(-np.pi/4, wires=target)
                qml.CNOT(wires=[ws[0], target])
                qml.RZ(np.pi/4, wires=target)
                qml.CNOT(wires=[ws[1], target])
                qml.RZ(-np.pi/4, wires=target)
                qml.Hadamard(wires=target)
            elif name == "CCZ":
                # Equivalence formula for CCZ
                qml.CNOT(wires=[ws[1], ws[2]])
                qml.RZ(-np.pi/4, wires=ws[2])
                qml.CNOT(wires=[ws[0], ws[2]])
                qml.RZ(np.pi/4, wires=ws[2])
                qml.CNOT(wires=[ws[1], ws[2]])
                qml.RZ(-np.pi/4, wires=ws[2])
                qml.CNOT(wires=[ws[0], ws[2]])
                qml.RZ(np.pi/4, wires=ws[1])
                qml.RZ(np.pi/4, wires=ws[2])
                qml.CNOT(wires=[ws[0], ws[1]])
                qml.RZ(np.pi/4, wires=ws[0])
                qml.RZ(-np.pi/4, wires=ws[1])
                qml.CNOT(wires=[ws[0], ws[1]])
            elif name == "CZ":
                ctrls, target = ws[:-1], ws[-1]
                qml.ctrl(qml.PauliZ, control=ctrls)(wires=target)
            elif name == "RZ":
                qml.RZ(params[0], wires=ws[0])
            elif name == "SWAP":
                qml.SWAP(wires=ws)
        return qml.state()

    return circuit


In [4]:
path = "datasets_Nam/Arithmetic_and_Toffoli/"
befores = sorted([f for f in os.listdir(path) if f.endswith("_before")])
after_ls = sorted([f for f in os.listdir(path) if f.endswith("_after_light")])
after_hs = sorted([f for f in os.listdir(path) if f.endswith("_after_heavy")])

In [5]:
print(len(befores))
print(len(after_ls))
print(len(after_hs))

29
29
29


In [6]:
len(befores)

29

In [7]:
idx = 6
before = quipper_to_qnode(path+befores[idx])
decomposed = quipper_to_qnode_extended(path+befores[idx])
after_l = quipper_to_qnode(path+after_ls[idx])
after_h = quipper_to_qnode(path+after_hs[idx])

# fig, ax = qml.draw_mpl(before)()
# plt.show()
# fig, ax = qml.draw_mpl(decomposed)()
# plt.show()
# fig, ax = qml.draw_mpl(after_l)()
# plt.show()
# fig, ax = qml.draw_mpl(after_h)()
# plt.show()

In [17]:
# summary
def spec_penny(circuit):
    obj = qml.specs(circuit)()['resources']
    return [obj.num_gates, obj.gate_sizes[1], obj.gate_sizes[2], obj.depth, list(obj.gate_types.keys())]

def spec_qiskit(qc):
    counts = {'1q': 0, '2q': 0}
    for gate in qc.data:
        if len(gate.qubits) == 1:
            counts['1q'] += 1
        elif len(gate.qubits) == 2:  
            counts['2q'] += 1
    return [qc.size(), counts['1q'], counts['2q'], qc.depth(), ]

df = pd.DataFrame(spec_penny(before), columns=['Original'])
df['Decomposed'] = spec_penny(decomposed)
df['Light'] = spec_penny(after_l)
df['Heavy'] = spec_penny(after_h)
df.index = ['Gate count', 'Single-qubit gate count', 'Two-qubit gate count','Circuit depth', 'Gate type']
df

,Original,Decomposed,Light,Heavy
Gate count,20,150,60,60
Single-qubit gate count,10,80,10,10
Two-qubit gate count,10,70,50,50
Circuit depth,5,84,38,38
Gate type,"[Hadamard, CNOT]","[Hadamard, CNOT, RZ]","[Hadamard, CNOT]","[Hadamard, CNOT]"


In [23]:
for idx, circ in enumerate(befores):
    print(str(circ.split('before')[0]))
    idx = idx
    before = quipper_to_qnode(path+befores[idx])
    decomposed = quipper_to_qnode_extended(path+befores[idx])
    after_l = quipper_to_qnode(path+after_ls[idx])
    after_h = quipper_to_qnode(path+after_hs[idx])

    df = pd.DataFrame(spec_penny(before), columns=['Original'])
    # df['Decomposed'] = spec_penny(decomposed)
    # df['Light'] = spec_penny(after_l)
    df['Heavy'] = spec_penny(after_h)
    df.index = ['Gate count', 'Single-qubit gate count', 'Two-qubit gate count','Circuit depth', 'Gate types']
    print(df)
    print()

adder_8_
                                         Original                     Heavy
Gate count                                    159                       375
Single-qubit gate count                        92                        84
Two-qubit gate count                           67                       291
Circuit depth                                  26                       121
Gate types               [CNOT, Hadamard, PauliX]  [CNOT, Hadamard, PauliX]

barenco_tof_10_
                           Original             Heavy
Gate count                       34               164
Single-qubit gate count          34                34
Two-qubit gate count              0               130
Circuit depth                     4               150
Gate types               [Hadamard]  [Hadamard, CNOT]

barenco_tof_3_
                           Original             Heavy
Gate count                        6                24
Single-qubit gate count           6                 6
Two-qubit gate c